[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C62_Coding_Interview_Course/04_dp_greedy/04_dp_greedy.ipynb)

# 04 · 动态规划与贪心（DP 五步 / 背包 / 交换论证 / NMS 的次优性）

目标：把 DP 从「背题解」变成**五句话的机械流程**，把贪心从「感觉可以」变成**能证明或能举反例**。

本 notebook 你会亲手实现：
1. **DP 五步检查器**，以及「答案位置取错」的翻车演示（Kadane）
2. **打家劫舍的两种状态定义**，构造出让「转移用 B、答案位置用 A」翻车的用例
3. **编辑距离**（二维 / 滚动数组 / **带状 O(nk)**）与 CER，**LCS 及方案回溯**
4. **01 背包与完全背包**：同一份一维代码，只把遍历方向反过来就变成另一道题
5. **LIS 的 O(n²) 与 O(n log n) 对拍**，并证明 `tails` 数组**不是**一个真实的上升子序列
6. **记忆化搜索 ↔ 递推**的互转与实测常数差
7. **交换论证的数值验证**：区间调度三种排序键谁对谁错；**找零的贪心反例**
8. **亮点实验**：把 NMS 建模成**加权区间调度**，用 DP 求最优解，
   量化贪心 NMS 的次优程度，并复现「一个大框吃掉两块牌子」的真实失效模式
9. 二维情形下最大权独立集的**暴力求解**，说明为什么工程上只能用贪心

> 心智模型：**状态定义是一句中文，不是一个数组。贪心要么能证明，要么能举反例，没有第三种答案。**

## 1 · DP 五步：先把五句话说出来

把五步写成一个 dict，逼自己每一步都填。**第 ⑤ 步「答案位置」是最容易漏的一项**——
所有中间值都算对，最后一行取错，是最难 debug 的一类错误。

In [ ]:
import bisect, math, random, time
from functools import lru_cache
import numpy as np

rng = np.random.default_rng(0)
random.seed(0)

FIVE_STEPS = ("状态定义", "转移方程", "初始化", "遍历顺序", "答案位置")

def check_dp_plan(plan):
    # 返回 (是否五步齐全, 缺失的步骤列表)
    missing = [s for s in FIVE_STEPS if not plan.get(s, "").strip()]
    return (len(missing) == 0), missing

plan_kadane = {
    "状态定义": "dp[i] = 以第 i 个元素结尾的最大子数组和",
    "转移方程": "dp[i] = max(dp[i-1] + a[i], a[i])   # 接上去 or 另起",
    "初始化":   "dp[0] = a[0]",
    "遍历顺序": "i 从小到大（dp[i] 依赖 dp[i-1]）",
    "答案位置": "max(dp)  —— 不是 dp[n-1]，因为最大子数组可以在任意位置结束",
}
plan_missing = dict(plan_kadane); plan_missing["答案位置"] = ""

print("完整计划 :", check_dp_plan(plan_kadane))
print("缺一步   :", check_dp_plan(plan_missing))
assert check_dp_plan(plan_kadane) == (True, [])
assert check_dp_plan(plan_missing) == (False, ["答案位置"])

# ── 按上面的五步实现 Kadane，并与暴力对拍 ──
def max_subarray(a):
    dp = best = a[0]                 # ③ 初始化
    for x in a[1:]:                  # ④ i 从小到大
        dp = max(dp + x, x)          # ② 转移
        best = max(best, dp)         # ⑤ 答案是 max(dp)
    return best

def max_subarray_last_dp(a):
    # 故意把 ⑤ 写成 dp[n-1]：中间值全对，只有答案位置错
    dp = a[0]
    for x in a[1:]:
        dp = max(dp + x, x)
    return dp

def max_subarray_brute(a):
    return max(sum(a[i:j]) for i in range(len(a)) for j in range(i + 1, len(a) + 1))

bad = 0
for _ in range(300):
    n = random.randint(1, 12)
    a = [random.randint(-9, 9) for _ in range(n)]
    assert max_subarray(a) == max_subarray_brute(a), a
    if max_subarray_last_dp(a) != max_subarray_brute(a):
        bad += 1
print(f"\n正确版 300/300 与暴力一致；\"答案位置\"写错的版本在 {bad}/300 个随机用例上给出错误答案")
demo = [-1, -2, -3]
print(f"最刺眼的一个: a={demo}  正确={max_subarray(demo)}  取 dp[n-1]={max_subarray_last_dp(demo)}")
assert max_subarray(demo) == -1 and max_subarray_last_dp(demo) == -3
assert bad > 50, "答案位置写错应该在相当比例的用例上翻车"
print("✅ 五步齐全 + Kadane 对拍通过。第 ⑤ 步不是形式主义。")

## 2 · 「状态定义错了后面全错」：打家劫舍的两种状态

同一道题（相邻两间不能同时偷，求最大值）有两种合法状态定义：

- **A**：`dp[i]` = 前 i 间房能偷到的最大值 → 答案 `dp[n]`
- **B**：`f[i]` = **必须偷第 i 间**时的最大值 → 答案 `max(f)`

两个都对。**但如果你用 B 的转移、A 的答案位置（`f[n-1]`），就会得到一个「大部分用例都对」的程序。**

In [ ]:
def rob_A(a):
    # ① dp[i] = 前 i 间的最大值   ⑤ 答案 dp[n]
    n = len(a)
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        take = a[i - 1] + (dp[i - 2] if i >= 2 else 0)
        dp[i] = max(dp[i - 1], take)
    return dp[n], dp

def rob_B_correct(a):
    # ① f[i] = **必须偷第 i 间**时的最大值   ⑤ 答案 max(f)（外加"一间都不偷"=0）
    n = len(a)
    f = [0] * n
    for i in range(n):
        f[i] = a[i] + max([f[j] for j in range(i - 1)] + [0])   # j <= i-2
    return max(f + [0]), f

def rob_B_wrong(a):
    # 转移用 B，答案位置照抄 A —— 典型的"状态定义与答案位置不匹配"
    _, f = rob_B_correct(a)
    return (f[-1] if f else 0), f

def rob_brute(a):
    n, best = len(a), 0
    for mask in range(1 << n):
        if mask & (mask >> 1):            # 有相邻两位同时为 1 -> 非法
            continue
        best = max(best, sum(a[i] for i in range(n) if mask >> i & 1))
    return best

trap = [5, 1, 1, 5, 1]
gA, dpA = rob_A(trap)
gB, fB = rob_B_correct(trap)
gW, _ = rob_B_wrong(trap)
print(f"a = {trap}")
print(f"  A 的 dp 表 = {dpA}      -> 答案 dp[n] = {gA}")
print(f"  B 的 f  表 = {fB}       -> 答案 max(f) = {gB}   /  错取 f[n-1] = {gW}")
print(f"  暴力      = {rob_brute(trap)}")
assert gA == gB == rob_brute(trap) == 10
assert gW == 7, gW      # 最优方案在第 3 间结束，f[n-1] 看不到它

n_wrong = 0
for _ in range(400):
    a = [random.randint(0, 9) for _ in range(random.randint(1, 12))]
    ref = rob_brute(a)
    assert rob_A(a)[0] == ref and rob_B_correct(a)[0] == ref, a
    n_wrong += (rob_B_wrong(a)[0] != ref)
print(f"\n错误版本在 {n_wrong}/400 个随机用例上出错 —— 也就是说约 {100*(1-n_wrong/400):.0f}% 的用例它是对的。")
print("✅ 这就是'状态定义错了后面全错'最阴险的地方：它不会在小用例上暴露。")
print("   诊断法：拿到错误答案先念状态定义那句话，再手算 dp[1]/dp[2] 核对，最后查答案位置。")

## 3 · 双序列 DP：编辑距离、带状加速与 LCS

编辑距离的三个分支就是**删 / 插 / 替换**。它同时是 OCR / 标志文字识别的 **CER** 指标的定义
（`CER = 编辑距离 / 参考长度`）。这里实现三个版本：二维表、滚动数组、**带状 O(nk)**（真实评测库的做法）。

In [ ]:
def edit_distance(a, b):
    n, m = len(a), len(b)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n + 1):
        D[i][0] = i                       # ③ 全删
    for j in range(m + 1):
        D[0][j] = j                       # ③ 全插
    for i in range(1, n + 1):             # ④ 依赖 左 / 上 / 左上
        for j in range(1, m + 1):
            D[i][j] = min(D[i - 1][j] + 1,                       # 删 a[i-1]
                          D[i][j - 1] + 1,                       # 插 b[j-1]
                          D[i - 1][j - 1] + (a[i - 1] != b[j - 1]))  # 匹配/替换
    return D[n][m]

def edit_distance_rolling(a, b):
    # 空间压缩到 O(min(n,m))：只保留上一行
    if len(a) < len(b):
        a, b = b, a
    prev = list(range(len(b) + 1))
    for i in range(1, len(a) + 1):
        cur = [i] + [0] * len(b)
        for j in range(1, len(b) + 1):
            cur[j] = min(prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return prev[len(b)]

def edit_distance_banded(a, b, k):
    # 只算 |i-j| <= k 的带状区域；若真实距离 > k 则返回 k+1（"至少 k+1"）
    n, m = len(a), len(b)
    INF = k + 1
    if abs(n - m) > k:
        return INF
    prev = [INF] * (m + 1)
    for j in range(0, min(m, k) + 1):
        prev[j] = j
    for i in range(1, n + 1):
        cur = [INF] * (m + 1)
        if i <= k:
            cur[0] = i
        for j in range(max(1, i - k), min(m, i + k) + 1):
            cur[j] = min(INF, prev[j] + 1, cur[j - 1] + 1,
                         prev[j - 1] + (a[i - 1] != b[j - 1]))
        prev = cur
    return min(prev[m], INF)

assert edit_distance("horse", "ros") == 3
assert edit_distance("intention", "execution") == 5
assert edit_distance("", "abc") == 3 and edit_distance("abc", "") == 3
assert edit_distance("abc", "abc") == 0

ALPHA = "abcd"
for _ in range(200):
    a = "".join(random.choice(ALPHA) for _ in range(random.randint(0, 9)))
    b = "".join(random.choice(ALPHA) for _ in range(random.randint(0, 9)))
    d = edit_distance(a, b)
    assert edit_distance_rolling(a, b) == d, (a, b)
    for k in (0, 1, 2, 3):
        assert min(edit_distance_banded(a, b, k), k + 1) == min(d, k + 1), (a, b, k)

# ── 用它算标志文字识别的 CER ──
def cer(ref, hyp):
    return edit_distance(ref, hyp) / max(1, len(ref))

pairs = [("限速120", "限速20"), ("限速60", "限速80"), ("前方施工", "前方施工"), ("禁止左转", "禁止右转")]
for r, h in pairs:
    print(f"  ref={r!r:>8}  hyp={h!r:>8}  编辑距离={edit_distance(r, h)}  CER={cer(r, h):.3f}")
assert abs(cer("限速120", "限速20") - 0.2) < 1e-12
assert cer("前方施工", "前方施工") == 0.0
print("✅ 编辑距离三版一致；带状版在 d<=k 时精确，是长序列 CER 的标准优化。")

In [ ]:
def lcs_table(a, b):
    n, m = len(a), len(b)
    L = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                L[i][j] = L[i - 1][j - 1] + 1
            else:
                L[i][j] = max(L[i - 1][j], L[i][j - 1])
    return L

def lcs_string(a, b):
    # 从表右下角回溯出一个具体方案 —— **空间压缩后就做不到这一步**
    L = lcs_table(a, b)
    i, j, out = len(a), len(b), []
    while i > 0 and j > 0:
        if a[i - 1] == b[j - 1]:
            out.append(a[i - 1]); i -= 1; j -= 1
        elif L[i - 1][j] >= L[i][j - 1]:
            i -= 1
        else:
            j -= 1
    return "".join(reversed(out))

def is_subseq(s, t):
    it = iter(t)
    return all(c in it for c in s)

A, B = "ABCBDAB", "BDCABA"
s = lcs_string(A, B)
print(f"LCS('{A}', '{B}') = '{s}'  长度 {len(s)}  (表值 {lcs_table(A, B)[len(A)][len(B)]})")
assert len(s) == 4 and lcs_table(A, B)[len(A)][len(B)] == 4
assert is_subseq(s, A) and is_subseq(s, B)

def lcs_brute(a, b):
    best = 0
    for mask in range(1 << len(a)):
        sub = "".join(a[i] for i in range(len(a)) if mask >> i & 1)
        if is_subseq(sub, b):
            best = max(best, len(sub))
    return best

for _ in range(150):
    a = "".join(random.choice("abc") for _ in range(random.randint(0, 8)))
    b = "".join(random.choice("abc") for _ in range(random.randint(0, 8)))
    got = lcs_table(a, b)[len(a)][len(b)]
    assert got == lcs_brute(a, b), (a, b)
    r = lcs_string(a, b)
    assert len(r) == got and is_subseq(r, a) and is_subseq(r, b)
print("✅ LCS 表值与暴力一致，且回溯出的串确实是两者的公共子序列。")
print("   记住：回溯需要完整的二维表 —— 这就是空间压缩的代价。")

## 4 · 背包：同一份代码，方向一反就是另一道题

`dp[c]` = 容量 c 下的最大价值。一维压缩后：

- **容量倒序** → 读到的 `dp[c-w]` 还是「上一行」的值 → 每件最多取 1 件 → **01 背包**
- **容量正序** → 读到的 `dp[c-w]` 已被本轮更新 → 同一件可重复取 → **完全背包**

In [ ]:
def knap01_2d(w, v, C):
    n = len(w)
    dp = [[0] * (C + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        for c in range(C + 1):
            dp[i][c] = dp[i - 1][c]
            if c >= w[i - 1]:
                dp[i][c] = max(dp[i][c], dp[i - 1][c - w[i - 1]] + v[i - 1])
    return dp[n][C]

def knap_1d(w, v, C, reverse=True):
    # reverse=True -> 01 背包；reverse=False -> 完全背包。**只有这一行的差别**
    dp = [0] * (C + 1)
    for wi, vi in zip(w, v):
        rng_c = range(C, wi - 1, -1) if reverse else range(wi, C + 1)
        for c in rng_c:
            dp[c] = max(dp[c], dp[c - wi] + vi)
    return dp[C]

def knap01_brute(w, v, C):
    n, best = len(w), 0
    for mask in range(1 << n):
        tw = sum(w[i] for i in range(n) if mask >> i & 1)
        if tw <= C:
            best = max(best, sum(v[i] for i in range(n) if mask >> i & 1))
    return best

W, V, C = [2, 3, 4], [3, 4, 6], 8
print(f"w={W} v={V} C={C}")
print(f"  01 背包  二维={knap01_2d(W, V, C)}  一维倒序={knap_1d(W, V, C, True)}  暴力={knap01_brute(W, V, C)}")
print(f"  完全背包 一维正序={knap_1d(W, V, C, False)}   (4 号物品拿两件: w=8, v=12)")
assert knap01_2d(W, V, C) == knap_1d(W, V, C, True) == knap01_brute(W, V, C) == 10
assert knap_1d(W, V, C, False) == 12

for _ in range(200):
    n = random.randint(1, 9)
    w = [random.randint(1, 7) for _ in range(n)]
    v = [random.randint(1, 20) for _ in range(n)]
    C = random.randint(0, 20)
    ref = knap01_brute(w, v, C)
    assert knap01_2d(w, v, C) == ref, (w, v, C)
    assert knap_1d(w, v, C, True) == ref, (w, v, C)
    assert knap_1d(w, v, C, False) >= ref            # 完全背包只会更大或相等
print("✅ 一维倒序 == 二维 == 暴力；正序则严格变成了完全背包。方向不是风格问题。")

# ── 工程化的用法：延迟预算下的模块配置选择（分组背包）──
BUDGET_TENTH_MS = 120                                   # 12.0 ms，量化到 0.1 ms
GROUPS = {                                              # 模块 -> [(名称, 延迟(0.1ms), AP 增益)]
    "detector": [("tiny", 35, 6.0), ("small", 55, 8.4), ("medium", 80, 9.6)],
    "tracker":  [("iou", 5, 0.8), ("kalman", 12, 1.4), ("bytetrack", 22, 1.9)],
    "classifier": [("none", 0, 0.0), ("cls-64", 18, 2.2), ("cls-128", 34, 3.1)],
}

def group_knapsack(groups, budget):
    NEG = -1e18
    dp = [NEG] * (budget + 1); dp[0] = 0.0
    pick = [[None] * (budget + 1) for _ in range(len(groups))]
    for gi, (gname, items) in enumerate(groups.items()):
        nd = [NEG] * (budget + 1)
        for c in range(budget + 1):
            for name, cost, gain in items:
                if cost <= c and dp[c - cost] > NEG / 2 and dp[c - cost] + gain > nd[c]:
                    nd[c] = dp[c - cost] + gain
                    pick[gi][c] = (name, cost)
        dp = nd
    best_c = max(range(budget + 1), key=lambda c: dp[c])
    plan, c = [], best_c
    for gi in range(len(groups) - 1, -1, -1):
        name, cost = pick[gi][c]
        plan.append((list(groups)[gi], name, cost / 10))
        c -= cost
    return dp[best_c], best_c / 10, list(reversed(plan))

gain, used, plan = group_knapsack(GROUPS, BUDGET_TENTH_MS)
print(f"\n预算 {BUDGET_TENTH_MS/10} ms -> 最优组合 AP 增益 {gain:.1f}，实际用掉 {used} ms")
for g, name, cost in plan:
    print(f"    {g:<11s} {name:<9s} {cost:.1f} ms")
assert used <= BUDGET_TENTH_MS / 10 + 1e-9
assert abs(gain - 13.7) < 1e-6, gain          # medium(8.0ms,9.6) + bytetrack(2.2,1.9) + cls-64(1.8,2.2)
print("✅ 「算力预算下选方案」= 分组背包。C57-05 / C63-04 的排序问题都是这个形状。")

In [ ]:
# ── 初始化陷阱：「恰好装满」必须用 inf，不能用 0 ──
def coin_min(coins, amount):
    INF = math.inf
    dp = [INF] * (amount + 1)
    dp[0] = 0                                   # ③ 只有 0 是天然可达的
    for c in range(1, amount + 1):
        for coin in coins:
            if coin <= c and dp[c - coin] + 1 < dp[c]:
                dp[c] = dp[c - coin] + 1
    return -1 if dp[amount] == INF else dp[amount]

def coin_min_zero_init(coins, amount):
    dp = [0] * (amount + 1)                     # ← 把不可达状态填成 0
    for c in range(1, amount + 1):
        for coin in coins:
            if coin <= c:
                dp[c] = min(dp[c], dp[c - coin] + 1)
    return dp[amount]

print("coins=[1,3,4]  amount=6 ->", coin_min([1, 3, 4], 6), "(3+3)")
print("coins=[2]      amount=3 ->", coin_min([2], 3), "(不可达)")
print("coins=[2]      amount=3 -> 0 初始化版本给出", coin_min_zero_init([2], 3), "  ← 谎报可行")
assert coin_min([1, 3, 4], 6) == 2
assert coin_min([1, 5, 10, 25], 63) == 6        # 25+25+10+1+1+1
assert coin_min([2], 3) == -1
assert coin_min_zero_init([2], 3) == 0          # 全 0 初始化 => 恒返回 0，错得毫无征兆
print("✅ 求 min 时不可达 = +inf，求 max 时 = -inf。填 0 等于宣称『存在一个代价 0 的方案』。")

## 5 · LIS：O(n²) 与 O(n log n) 对拍，以及 `tails` 的真相

`tails[k]` = **所有长度为 k+1 的上升子序列中最小的那个结尾值**。
它单调递增所以能二分，但**它的内容不是任何一个真实的 LIS**——只有长度是对的。
这是这道题面试中最经典的误解。

In [ ]:
def lis_n2(a):
    if not a:
        return 0
    dp = [1] * len(a)                       # ① dp[i] = **以 i 结尾**的 LIS 长度
    for i in range(len(a)):
        for j in range(i):
            if a[j] < a[i]:
                dp[i] = max(dp[i], dp[j] + 1)
    return max(dp)                          # ⑤ 答案是 max(dp)

def lis_nlogn(a, return_tails=False):
    tails = []
    for x in a:
        i = bisect.bisect_left(tails, x)    # 严格上升用 bisect_left
        if i == len(tails):
            tails.append(x)
        else:
            tails[i] = x                    # 用更小的结尾替换，为将来留余地
    return (len(tails), tails) if return_tails else len(tails)

for _ in range(300):
    a = [random.randint(0, 15) for _ in range(random.randint(0, 14))]
    assert lis_n2(a) == lis_nlogn(a), a
print("O(n^2) 与 O(n log n) 在 300 个随机数组上完全一致")

a = [1, 3, 5, 2]
L, tails = lis_nlogn(a, return_tails=True)
print(f"\na = {a}   LIS 长度 = {L}   tails = {tails}")
assert tails == [1, 2, 5] and L == 3
assert not is_subseq(tails, a), "本例中 tails 恰恰不是 a 的子序列"
print(f"  tails={tails} 在 a 里？ {is_subseq(tails, a)}   ← 5 出现在 2 之前，tails 不是合法子序列")
print("  真实的一个 LIS 是 [1,3,5]。要还原具体序列必须额外记录前驱下标。")

# 计时对比：n 大时 O(n^2) 会明显吃亏
for n in (400, 1600):
    big = [random.randint(0, 10**6) for _ in range(n)]
    t0 = time.perf_counter(); r1 = lis_n2(big);   t1 = time.perf_counter()
    r2 = lis_nlogn(big);                          t2 = time.perf_counter()
    assert r1 == r2
    print(f"  n={n:>5d}  O(n^2) {1e3*(t1-t0):7.2f} ms   O(n log n) {1e3*(t2-t1):7.2f} ms")
print("✅ 面试里先写 O(n^2) 保证正确，再说 O(n log n) 的 tails 语义 —— 顺序别反。")

## 6 · 记忆化搜索 ↔ 递推：先写能跑的那一版

同一道 01 背包，两种写法。**记忆化把「遍历顺序」这一步免掉了**，代价是常数与栈深度。

In [ ]:
def knap_memo(w, v, C):
    w, v = tuple(w), tuple(v)

    @lru_cache(maxsize=None)
    def f(i, c):                     # f(i,c) = 只考虑前 i 件、容量 c 时的最大价值
        if i == 0 or c == 0:
            return 0
        best = f(i - 1, c)                                  # 不选第 i 件
        if w[i - 1] <= c:
            best = max(best, f(i - 1, c - w[i - 1]) + v[i - 1])   # 选第 i 件
        return best

    r = f(len(w), C)
    return r, f.cache_info()

n = 60
W = [25 * random.randint(1, 12) for _ in range(n)]   # 延迟以 25 微秒为量化单位 -> 状态空间稀疏
V = [random.randint(1, 100) for _ in range(n)]
CAP = 750

t0 = time.perf_counter(); r_memo, info = knap_memo(W, V, CAP); t1 = time.perf_counter()
r_tab = knap_1d(W, V, CAP, True);                              t2 = time.perf_counter()
print(f"记忆化搜索 = {r_memo}   耗时 {1e3*(t1-t0):.1f} ms   缓存命中/未命中 = {info.hits}/{info.misses}")
print(f"自底向上   = {r_tab}   耗时 {1e3*(t2-t1):.1f} ms")
assert r_memo == r_tab
full_table = (n + 1) * (CAP + 1)
print(f"完整状态表 {full_table} 格，记忆化只算了 {info.misses} 格（{info.misses/full_table:.1%}）"
      " ← 稀疏状态空间正是记忆化的优势")
assert info.misses < full_table, "记忆化应当只触及可达状态"
print()
print("互转口诀:  函数参数 -> 数组下标 | 递归调用 -> 读更早的元素 | 边界 -> 初始化 | (递归不用想) -> 遍历顺序")
print("注意: Python 默认递归深度 1000，链式依赖的题（如 n=1e4 的 LIS）用记忆化会 RecursionError。")
print("✅ 面试策略：暴力递归 -> 加 @lru_cache -> （口头）改递推 + 压缩空间。")

## 7 · 贪心：交换论证的数值验证，和一个「看起来对但错」的反例

区间调度里三种「看起来都合理」的排序键：**最早结束 / 最早开始 / 最短区间**。
只有第一种能通过交换论证，另外两种在随机用例上会被打脸。

In [ ]:
def overlap(a, b):
    # 半开区间 [s,e)：接触不算冲突
    return min(a[1], b[1]) - max(a[0], b[0]) > 0

def schedule_greedy(iv, key):
    keep = []
    for i in sorted(range(len(iv)), key=lambda i: key(iv[i])):
        if all(not overlap(iv[i], iv[j]) for j in keep):
            keep.append(i)
    return sorted(keep)

def schedule_brute(iv):
    n, best = len(iv), []
    for mask in range(1 << n):
        sel = [i for i in range(n) if mask >> i & 1]
        if all(not overlap(iv[a], iv[b]) for a in sel for b in sel if a < b):
            if len(sel) > len(best):
                best = sel
    return best

KEYS = {
    "最早结束 (正确)": lambda x: x[1],
    "最早开始":        lambda x: x[0],
    "最短区间":        lambda x: x[1] - x[0],
}
fails = {k: 0 for k in KEYS}
for _ in range(400):
    n = random.randint(1, 9)
    iv = []
    for _ in range(n):
        s = random.randint(0, 20); iv.append((s, s + random.randint(1, 8)))
    opt = len(schedule_brute(iv))
    for k, f in KEYS.items():
        if len(schedule_greedy(iv, f)) != opt:
            fails[k] += 1
for k, c in fails.items():
    print(f"  {k:<16s} 在 400 组随机用例中失手 {c} 次")
assert fails["最早结束 (正确)"] == 0, "最早结束贪心必须永远最优（交换论证）"
assert fails["最早开始"] > 0 and fails["最短区间"] > 0
print()
print("交换论证（最早结束）: 设最优解 O 与贪心解 G 第一次在第 k 步不同，g_k 的结束时间 e <= o_k 的 e'，")
print("  把 o_k 换成 g_k 后，O 的第 k+1 个区间起点 >= e' >= e，仍不冲突，个数不变 => O' 同样最优。")
demo = [(0, 10), (1, 2), (3, 4)]
print(f"\n最早开始的反例: {demo} -> 它选 {schedule_greedy(demo, KEYS['最早开始'])}，最优是 {schedule_brute(demo)}")
demo2 = [(0, 4), (3, 5), (4, 8)]
print(f"最短区间的反例: {demo2} -> 它选 {schedule_greedy(demo2, KEYS['最短区间'])}，最优是 {schedule_brute(demo2)}")
assert len(schedule_greedy(demo, KEYS["最早开始"])) == 1 and len(schedule_brute(demo)) == 2
assert len(schedule_greedy(demo2, KEYS["最短区间"])) == 1 and len(schedule_brute(demo2)) == 2
print("✅ 「排序键」选错，贪心就错。能证明的只有最早结束这一个。")

In [ ]:
# ── 找零：贪心在某些面值系上正确，在另一些上错 —— 而反例往往不在最小的用例里 ──
def coin_greedy(coins, amount):
    left, cnt = amount, 0
    for c in sorted(coins, reverse=True):
        take = left // c
        cnt += take; left -= take * c
    return cnt if left == 0 else -1

def scan_counterexamples(coins, hi=40):
    out = []
    for amt in range(1, hi + 1):
        g, d = coin_greedy(coins, amt), coin_min(coins, amt)
        if d != -1 and (g == -1 or g > d):
            out.append((amt, g, d))
    return out

for coins in ([1, 3, 4], [1, 15, 25], [1, 5, 10, 25], [1, 5, 10, 20, 50, 100]):
    ce = scan_counterexamples(coins, 40)
    head = ce[:4]
    print(f"  面值 {str(coins):<26s} 反例数 {len(ce):>2d}  前几个 (金额, 贪心, 最优): {head}")

ce134 = scan_counterexamples([1, 3, 4], 40)
assert (6, 3, 2) in ce134, ce134[:5]
assert coin_greedy([1, 3, 4], 5) == coin_min([1, 3, 4], 5) == 2, "amount=5 上贪心恰好对 —— 这就是陷阱"
assert scan_counterexamples([1, 5, 10, 25], 100) == []
assert scan_counterexamples([1, 5, 10, 20, 50, 100], 200) == []
print()
print("陷阱演示: amount=5 时贪心 4+1=2 枚 == 最优；amount=6 时贪心 4+1+1=3 枚 > 最优 3+3=2 枚。")
print("✅ 只手测一两个小用例就宣称『贪心可以』，是面试里最常见的失分方式。")

## 8 · 亮点实验：NMS = 加权区间调度的贪心近似

一维场景（龙门架上并排的标志）里，把检测框看成区间、score 看成权重、IoU 阈值取 0，于是：

| | NMS | 加权区间调度 |
|---|---|---|
| 约束 | 保留的框两两不重叠 | 选中的区间两两不重叠 |
| 算法 | 按分数降序，能留就留（**贪心**） | 按结束时间排序 + **DP** |
| 最优性 | 无保证 | 精确最优 |

`OPT(j) = max(OPT(j-1), w_j + OPT(p(j)))`，其中 `p(j)` = 最大的 `i<j` 使 `end_i <= start_j`。

In [ ]:
def greedy_nms_1d(iv, w):
    # 标准贪心 NMS（IoU 阈值 = 0：任何重叠都抑制）
    keep = []
    for i in sorted(range(len(iv)), key=lambda i: -w[i]):
        if all(not overlap(iv[i], iv[j]) for j in keep):
            keep.append(i)
    return sorted(keep)

def weighted_interval_scheduling(iv, w):
    # 返回 (最优总权重, 选中的原始下标列表)
    n = len(iv)
    if n == 0:
        return 0.0, []
    idx = sorted(range(n), key=lambda i: iv[i][1])       # 按结束时间升序
    ends = [iv[i][1] for i in idx]
    p = [bisect.bisect_right(ends, iv[idx[k]][0], 0, k) for k in range(n)]
    OPT = [0.0] * (n + 1)
    take_it = [False] * (n + 1)
    for k in range(1, n + 1):
        take = w[idx[k - 1]] + OPT[p[k - 1]]
        if take > OPT[k - 1] + 1e-12:
            OPT[k], take_it[k] = take, True
        else:
            OPT[k] = OPT[k - 1]
    sel, k = [], n
    while k > 0:
        if take_it[k]:
            sel.append(idx[k - 1]); k = p[k - 1]
        else:
            k -= 1
    return OPT[n], sorted(sel)

# ── 教科书级的三框反例：一个大框 vs 两个小框 ──
IV = [(0, 10), (0, 4), (6, 10)]
WT = [10.0, 6.0, 6.0]
g = greedy_nms_1d(IV, WT); gw = sum(WT[i] for i in g)
ow, o = weighted_interval_scheduling(IV, WT)
print("      A ████████████████████  w=10")
print("      B █████                 w=6")
print("      C           ██████      w=6")
print(f"\n贪心 NMS 保留 {['ABC'[i] for i in g]}  总分 {gw}")
print(f"DP  最优 保留 {['ABC'[i] for i in o]}  总分 {ow}")
print(f"贪心 / 最优 = {gw/ow:.4f}")
assert g == [0] and gw == 10.0
assert o == [1, 2] and ow == 12.0
assert abs(gw / ow - 10 / 12) < 1e-12

# ── DP 的正确性：与暴力枚举对拍 ──
def wis_brute(iv, w):
    n, best = len(iv), 0.0
    for mask in range(1 << n):
        sel = [i for i in range(n) if mask >> i & 1]
        if all(not overlap(iv[a], iv[b]) for a in sel for b in sel if a < b):
            best = max(best, sum(w[i] for i in sel))
    return best

for _ in range(300):
    n = random.randint(0, 10)
    iv, w = [], []
    for _ in range(n):
        s = random.randint(0, 30); iv.append((s, s + random.randint(1, 9)))
        w.append(round(random.uniform(0.1, 1.0), 3))
    ow2, sel = weighted_interval_scheduling(iv, w)
    assert abs(ow2 - wis_brute(iv, w)) < 1e-9, (iv, w)
    assert all(not overlap(iv[a], iv[b]) for a in sel for b in sel if a < b)
    assert abs(sum(w[i] for i in sel) - ow2) < 1e-9
print("✅ 加权区间调度 DP 与暴力枚举在 300 组随机用例上完全一致（且方案确实互不重叠）。")

In [ ]:
# ── 贪心到底能差多少？答案：可以任意差 ──
def adversarial(k, eps=0.01):
    # 一个覆盖 k 块牌子的大框（权重 1+eps） + k 个正确的小框（权重各 1）
    iv = [(0.0, float(k))] + [(float(i), float(i + 1)) for i in range(k)]
    w = [1.0 + eps] + [1.0] * k
    return iv, w

print("最坏情况族：一个略高分的大框 vs k 个正确的小框")
print(f"{'k':>4s} {'贪心':>8s} {'最优':>8s} {'比值':>8s}")
ratios = []
for k in (2, 5, 10, 50, 200):
    iv, w = adversarial(k)
    gw = sum(w[i] for i in greedy_nms_1d(iv, w))
    ow, _ = weighted_interval_scheduling(iv, w)
    ratios.append(gw / ow)
    print(f"{k:>4d} {gw:>8.2f} {ow:>8.2f} {gw/ow:>8.4f}")
assert all(ratios[i] > ratios[i + 1] for i in range(len(ratios) - 1))
assert ratios[-1] < 0.01, "k=200 时比值应趋于 0 —— 贪心近似比无常数下界"
print("=> 贪心 NMS 的近似比**没有常数下界**，不是『差一点点』。")

# ── 随机场景上的统计：平均很好，但确实存在次优 ──
def random_scene(n, rng):
    s = rng.uniform(0, 100, size=n)
    e = s + rng.uniform(2, 18, size=n)
    w = rng.uniform(0.1, 1.0, size=n)
    return list(zip(s.tolist(), e.tolist())), w.tolist()

rs = np.random.default_rng(7)
rr, n_sub = [], 0
for _ in range(400):
    iv, w = random_scene(rs.integers(4, 16), rs)
    gw = sum(w[i] for i in greedy_nms_1d(iv, w))
    ow, _ = weighted_interval_scheduling(iv, w)
    rr.append(gw / ow)
    n_sub += (gw < ow - 1e-9)
rr = np.array(rr)
print(f"\n随机场景 400 组: 平均比值 {rr.mean():.4f}  最差 {rr.min():.4f}  "
      f"5% 分位 {np.percentile(rr,5):.4f}  贪心次优的比例 {n_sub/400:.1%}")
assert 0 < n_sub < 400
assert rr.mean() > 0.85 and rr.min() < 1.0
print("✅ 平均场景下贪心很接近最优，**但最坏情况恰好发生在密集排列的标志上** —— 而那正是 TSR 最在意的场景。")

In [ ]:
# ── 真实失效模式：一个"合并框"吃掉两块相邻的牌子 ──
def iou_1d(a, b):
    inter = max(0.0, min(a[1], b[1]) - max(a[0], b[0]))
    union = (a[1] - a[0]) + (b[1] - b[0]) - inter
    return inter / union if union > 0 else 0.0

def build_gantry(n_pairs=4):
    # 每对: 两块相邻的牌子 + 各自的正确框 + 一个跨越两者的"合并"误检（分数略高）
    gts, iv, w, tag = [], [], [], []
    for t in range(n_pairs):
        b = 100.0 * t
        g1, g2 = (b, b + 10), (b + 14, b + 24)
        gts += [g1, g2]
        iv += [g1, g2, (b, b + 24)]
        w += [0.60, 0.55, 0.63]
        tag += ["正确框1", "正确框2", "合并误检"]
    return gts, iv, w, tag

def recall_at(gts, iv, keep, thr=0.5):
    hit = sum(any(iou_1d(g, iv[i]) >= thr for i in keep) for g in gts)
    return hit / len(gts)

gts, iv, w, tag = build_gantry(4)
kg = greedy_nms_1d(iv, w)
ow, ko = weighted_interval_scheduling(iv, w)
gw = sum(w[i] for i in kg)
print(f"场景: {len(gts)} 块牌子, {len(iv)} 个候选框（每对牌子附带一个跨越两者的合并误检 score=0.63）")
print(f"  贪心 NMS 保留 {len(kg)} 个: {sorted(set(tag[i] for i in kg))}   总分 {gw:.2f}   召回 {recall_at(gts,iv,kg):.0%}")
print(f"  DP  最优 保留 {len(ko)} 个: {sorted(set(tag[i] for i in ko))}   总分 {ow:.2f}   召回 {recall_at(gts,iv,ko):.0%}")
print(f"  合并框与单块牌子的 IoU = {iou_1d((0,24),(0,10)):.4f} < 0.5 -> 它一个牌子都算不上检出")
assert recall_at(gts, iv, kg) == 0.0 and recall_at(gts, iv, ko) == 1.0
assert abs(gw - 4 * 0.63) < 1e-9 and abs(ow - 4 * 1.15) < 1e-9
assert all(tag[i] == "合并误检" for i in kg)
print("✅ 贪心 NMS 的失效模式可以被精确刻画：**高分框独吞一片区域**。")
print("   这正是 Soft-NMS（衰减而非删除）、WBF（融合而非择一）、")
print("   以及 DETR/YOLOv10 的一对一分配（训练期就去重）想解决的同一个问题。")

In [ ]:
# ── 那为什么工业界不换成 DP？因为二维情形下问题变成 NP-hard ──
def iou_2d(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def nms_2d(boxes, scores, thr=0.5):
    keep = []
    for i in sorted(range(len(boxes)), key=lambda i: -scores[i]):
        if all(iou_2d(boxes[i], boxes[j]) <= thr for j in keep):
            keep.append(i)
    return sorted(keep)

def mwis_brute_2d(boxes, scores, thr=0.5):
    # 冲突图上的最大权独立集：**只能枚举 2^n**
    n = len(boxes)
    adj = [0] * n
    for i in range(n):
        for j in range(i + 1, n):
            if iou_2d(boxes[i], boxes[j]) > thr:
                adj[i] |= 1 << j; adj[j] |= 1 << i
    best, bmask = 0.0, 0
    for mask in range(1 << n):
        ok, m = True, mask
        while m:
            i = (m & -m).bit_length() - 1
            if adj[i] & mask:
                ok = False; break
            m &= m - 1
        if ok:
            s = sum(scores[i] for i in range(n) if mask >> i & 1)
            if s > best:
                best, bmask = s, mask
    return best, sorted(i for i in range(n) if bmask >> i & 1)

r2 = np.random.default_rng(11)
worse, tot = 0, 0
for _ in range(60):
    n = int(r2.integers(6, 13))
    cx, cy = r2.uniform(0, 40, n), r2.uniform(0, 40, n)
    ww, hh = r2.uniform(8, 26, n), r2.uniform(8, 26, n)
    boxes = [(cx[i] - ww[i] / 2, cy[i] - hh[i] / 2, cx[i] + ww[i] / 2, cy[i] + hh[i] / 2) for i in range(n)]
    sc = r2.uniform(0.1, 1.0, n).tolist()
    g = sum(sc[i] for i in nms_2d(boxes, sc))
    o, _ = mwis_brute_2d(boxes, sc)
    assert o >= g - 1e-9, "暴力最优不可能比贪心差"
    worse += (g < o - 1e-9); tot += 1
print(f"二维随机场景 {tot} 组：贪心 NMS 严格次优的比例 {worse/tot:.0%}")

print("\n暴力 MWIS 的耗时随 n 指数增长:")
for n in (10, 13, 16):
    cx, cy = r2.uniform(0, 60, n), r2.uniform(0, 60, n)
    boxes = [(cx[i] - 10, cy[i] - 10, cx[i] + 10, cy[i] + 10) for i in range(n)]
    sc = r2.uniform(0.1, 1.0, n).tolist()
    t0 = time.perf_counter(); mwis_brute_2d(boxes, sc); dt = time.perf_counter() - t0
    print(f"  n={n:>3d}  2^n={2**n:>8d}  {1e3*dt:8.1f} ms")
print("  一帧检测器输出常有 N=1000+ 个过阈框 -> 2^1000 完全不可行。")
print("✅ 结论：**一维退化情形可以 DP 求最优；二维是 NP-hard，只能贪心 + 工程补丁。**")
print("   面试标准答案：『NMS 是最大权独立集的贪心近似；我可以在一维情形下用 DP 求最优并量化 gap，")
print("   但二维不可行，工程上的解法是 Soft-NMS / WBF / 一对一标签分配。』")

## ✏️ 练习 1：最长公共子序列（高频 · 必会 · 15 行以内）

实现 `lcs_length(a, b)`，返回两个字符串的 LCS 长度。

**动手前先把五步说出来**：
① `L[i][j]` = a 前 i 个与 b 前 j 个的 LCS 长度；② 相等则 `L[i-1][j-1]+1`，否则 `max(L[i-1][j], L[i][j-1])`；
③ `L[0][*]=L[*][0]=0`；④ i、j 都从小到大；⑤ 答案 `L[n][m]`。

In [ ]:
def lcs_length(a, b):
    # TODO: 二维 DP，返回 LCS 长度
    raise NotImplementedError

In [ ]:
# —— 练习 1 自测 ——
assert lcs_length("ABCBDAB", "BDCABA") == 4
assert lcs_length("abcde", "ace") == 3
assert lcs_length("abc", "") == 0 and lcs_length("", "") == 0
assert lcs_length("abc", "def") == 0
assert lcs_length("aaa", "aa") == 2
for _ in range(200):
    a = "".join(random.choice("abc") for _ in range(random.randint(0, 8)))
    b = "".join(random.choice("abc") for _ in range(random.randint(0, 8)))
    assert lcs_length(a, b) == lcs_brute(a, b), (a, b)
print("✅ 练习 1 通过：与暴力枚举在 200 组随机串上一致。")
print("   面试加分句：『空间可压到 O(min(n,m))，但那样就回溯不出具体的公共子序列了。』")

## ✏️ 练习 2：01 背包的一维空间压缩（高频 · 必会）

实现 `knapsack01_1d(weights, values, C)`，**必须用一维数组**，返回最大价值。

**考点只有一个：遍历容量的方向。**方向反了就变成完全背包，而自测会当场抓住你。

In [ ]:
def knapsack01_1d(weights, values, C):
    # TODO: dp = [0]*(C+1)，对每件物品遍历容量 —— 方向？
    raise NotImplementedError

In [ ]:
# —— 练习 2 自测 ——
assert knapsack01_1d([2, 3, 4], [3, 4, 6], 8) == 10
assert knapsack01_1d([], [], 10) == 0
assert knapsack01_1d([5], [9], 4) == 0            # 装不下
assert knapsack01_1d([5], [9], 5) == 9
# 关键：必须**严格不等于**完全背包的答案，否则说明方向写反了
assert knapsack01_1d([2, 3, 4], [3, 4, 6], 8) != knap_1d([2, 3, 4], [3, 4, 6], 8, reverse=False)
for _ in range(200):
    n = random.randint(0, 9)
    w = [random.randint(1, 7) for _ in range(n)]
    v = [random.randint(1, 20) for _ in range(n)]
    C = random.randint(0, 20)
    assert knapsack01_1d(w, v, C) == knap01_brute(w, v, C), (w, v, C)
print("✅ 练习 2 通过：一维压缩 + 容量倒序 == 二维 == 暴力。")
print("   面试加分句：『倒序是因为 dp[c-w] 必须还是上一行的值——否则同一件物品会被重复选。』")

## ✏️ 练习 3：加权区间调度 DP（低频 · 加分 · 本模块的核心）

实现 `weighted_interval_dp(intervals, weights)`，返回**最优总权重**（float）。

区间是半开的 `[s, e)`，接触不算重叠。提示：按结束时间排序 → 用二分求 `p(j)` → `OPT(j)=max(OPT(j-1), w_j+OPT(p(j)))`。

In [ ]:
def weighted_interval_dp(intervals, weights):
    # TODO: 返回最大总权重（不需要返回方案）
    raise NotImplementedError

In [ ]:
# —— 练习 3 自测 ——
assert abs(weighted_interval_dp([(0, 10), (0, 4), (6, 10)], [10.0, 6.0, 6.0]) - 12.0) < 1e-9
assert weighted_interval_dp([], []) == 0
assert abs(weighted_interval_dp([(0, 5)], [3.0]) - 3.0) < 1e-9
assert abs(weighted_interval_dp([(0, 5), (5, 10)], [1.0, 2.0]) - 3.0) < 1e-9   # 半开区间：接触不冲突
assert abs(weighted_interval_dp([(0, 5), (4, 10)], [1.0, 2.0]) - 2.0) < 1e-9   # 真重叠：只能取一个
for _ in range(300):
    n = random.randint(0, 10)
    iv, w = [], []
    for _ in range(n):
        s = random.randint(0, 30); iv.append((s, s + random.randint(1, 9)))
        w.append(round(random.uniform(0.1, 1.0), 3))
    assert abs(weighted_interval_dp(iv, w) - wis_brute(iv, w)) < 1e-9, (iv, w)
print("✅ 练习 3 通过：与 2^n 暴力枚举在 300 组随机用例上一致。")
print("   面试加分句：『p(j) 用二分求，所以总复杂度是排序主导的 O(n log n)。』")

## ✏️ 练习 4：量化贪心 NMS 的次优程度

实现 `greedy_gap(intervals, weights)`，返回 `(greedy_total, opt_total, ratio)`：

- `greedy_total`：贪心 NMS（按权重降序，能留就留）保留下来的总权重
- `opt_total`：加权区间调度 DP 的最优总权重
- `ratio`：`greedy_total / opt_total`（`opt_total == 0` 时返回 `1.0`）

可以直接复用前面的 `greedy_nms_1d` 与你在练习 3 写的 `weighted_interval_dp`。

In [ ]:
def greedy_gap(intervals, weights):
    # TODO: 返回 (greedy_total, opt_total, ratio)
    raise NotImplementedError

In [ ]:
# —— 练习 4 自测 ——
g, o, r = greedy_gap([(0, 10), (0, 4), (6, 10)], [10.0, 6.0, 6.0])
assert abs(g - 10.0) < 1e-9 and abs(o - 12.0) < 1e-9 and abs(r - 10 / 12) < 1e-9
assert greedy_gap([], []) == (0, 0, 1.0) or abs(greedy_gap([], [])[2] - 1.0) < 1e-12
iv_a, w_a = adversarial(10)
g, o, r = greedy_gap(iv_a, w_a)
assert abs(g - 1.01) < 1e-9 and abs(o - 10.0) < 1e-9 and r < 0.11, (g, o, r)
# 贪心永远不会优于最优；随机场景下平均很接近 1 但确实存在次优
rr4 = np.random.default_rng(23)
vals, sub = [], 0
for _ in range(300):
    iv, w = random_scene(int(rr4.integers(4, 16)), rr4)
    g, o, r = greedy_gap(iv, w)
    assert r <= 1.0 + 1e-9, r
    vals.append(r); sub += (r < 1 - 1e-9)
print(f"随机 300 组：平均比值 {np.mean(vals):.4f}  最差 {min(vals):.4f}  次优比例 {sub/300:.1%}")
assert 0 < sub < 300 and np.mean(vals) > 0.85
print("✅ 练习 4 通过：贪心 NMS 的 gap 是可以被量化的，而它的最坏情形没有常数下界。")

---
### 📖 参考答案（先自己做，再对照）

In [ ]:
# 练习 1 参考答案
def lcs_length(a, b):
    n, m = len(a), len(b)
    L = [[0] * (m + 1) for _ in range(n + 1)]          # ③ 边界天然是 0
    for i in range(1, n + 1):                          # ④ 依赖 左 / 上 / 左上
        for j in range(1, m + 1):
            if a[i - 1] == b[j - 1]:
                L[i][j] = L[i - 1][j - 1] + 1          # ② 配上了
            else:
                L[i][j] = max(L[i - 1][j], L[i][j - 1])  # ② 丢掉 a 的最后一个 或 b 的最后一个
    return L[n][m]                                     # ⑤

In [ ]:
# 练习 2 参考答案
def knapsack01_1d(weights, values, C):
    dp = [0] * (C + 1)
    for wi, vi in zip(weights, values):
        for c in range(C, wi - 1, -1):     # **倒序**：保证 dp[c-wi] 还是"没考虑本件"的值
            dp[c] = max(dp[c], dp[c - wi] + vi)
    return dp[C]

In [ ]:
# 练习 3 参考答案
def weighted_interval_dp(intervals, weights):
    n = len(intervals)
    if n == 0:
        return 0
    idx = sorted(range(n), key=lambda i: intervals[i][1])       # 按结束时间升序
    ends = [intervals[i][1] for i in idx]
    OPT = [0.0] * (n + 1)
    for k in range(1, n + 1):
        s = intervals[idx[k - 1]][0]
        p = bisect.bisect_right(ends, s, 0, k - 1)              # 最后一个 end <= s 的位置数
        OPT[k] = max(OPT[k - 1], weights[idx[k - 1]] + OPT[p])
    return OPT[n]

In [ ]:
# 练习 4 参考答案
def greedy_gap(intervals, weights):
    g = sum(weights[i] for i in greedy_nms_1d(intervals, weights))
    o = weighted_interval_dp(intervals, weights)
    return g, o, (g / o if o > 0 else 1.0)

---
## 🧪 真实工程胶囊：把工程问题归约到 DP / 贪心的检查单

In [ ]:
RECIPE = r'''
# ══════════════════════════════════════════════════════════════════════
# A. 白板上的 DP 五步（每一步都要出声说）
# ══════════════════════════════════════════════════════════════════════
# ① 状态定义  "dp[i] 表示 ___"          <- 必须含「以 i 结尾」或「前 i 个」这类限定词
# ② 转移方程  枚举"最后一步的决策"，每个决策一个分支
# ③ 初始化    不可达用 -inf(求 max) / +inf(求 min)，**不要填 0**
# ④ 遍历顺序  画依赖箭头；一维压缩时：01 背包倒序、完全背包正序、区间 DP 按长度
# ⑤ 答案位置  dp[n] ? max(dp) ? dp[n][m] ?  <- 把①那句话代入题目问句读一遍
#
# 卡住时的脱困路径（说出来就得分）：
#   暴力递归（保证语义对） -> @lru_cache（变成 DP） -> 递推 + 空间压缩（如果还有时间）

# ══════════════════════════════════════════════════════════════════════
# B. 贪心：要么证明，要么反例，不许说"我觉得"
# ══════════════════════════════════════════════════════════════════════
# 交换论证模板：
#   设最优解 O 与贪心解 G 在第 k 步第一次不同（前 k-1 步相同）
#   把 O 的第 k 步换成 G 的第 k 步得到 O'
#   证明 O' 仍合法 且 目标值不下降  =>  G 也最优
# 如果第 3 步走不通（换完不合法/更差），这本身就是"贪心不对"的强信号 -> 改 DP
# 反例构造清单（60 秒内试完）：
#   · 一个大项 vs 多个小项（找零 [1,3,4] 凑 6；加权区间调度）
#   · 局部最优挤掉后续机会（区间调度按最早开始 / 最短区间排序）
#   · 边界：空 / 单元素 / 全相同 / 恰好等于阈值

# ══════════════════════════════════════════════════════════════════════
# C. 检测后处理：NMS 是贪心近似，知道它错在哪就知道该换什么
# ══════════════════════════════════════════════════════════════════════
# 失效模式：**高分框独吞一片区域** —— 一个跨越两块牌子的合并框把两个正确框都杀了
# 对应的工程补丁（按代价从低到高）：
#   1) 调 iou_thr           密集场景调高（0.5 -> 0.65），代价是重复框变多
#   2) class-aware NMS      按类别分组做，避免不同类互相抑制（限速牌 vs 指路牌）
#   3) Soft-NMS             不删框，按重叠度衰减分数：s_i *= exp(-IoU^2 / sigma)
#                           sigma 典型 0.5；只在离线/评测用，车端慎用（保留框数变多 -> 延迟涨）
#   4) WBF (weighted boxes fusion)   一簇框加权融合而非择一，见 C56-04；适合 TTA / 多模型集成
#   5) 一对一标签分配        DETR 系匈牙利匹配 / YOLOv10 一致双分配 —— 训练期去重，
#                           推理端**根本不需要 NMS**（见 C53-04、C54）
#
# ── 推荐的验证方法（可直接抄进你的评测脚本）──
#   · 造一维退化用例（龙门架并排标志），用加权区间调度 DP 求最优，量化 NMS 的 gap
#   · 按"每个 GT 被几个保留框覆盖"分桶统计：>1 是重复，=0 是漏检，定位失效类型
#   · 延迟：NMS 是 O(N·K)，N 与 K 都随场景增长 -> 报 p50 和 **p99** 两个数（见 C53-04）

# ══════════════════════════════════════════════════════════════════════
# D. 预算分配 = 分组背包（把工程问题写成 DP 的最常见入口）
# ══════════════════════════════════════════════════════════════════════
#   把连续预算（ms / GB / 人天）量化成整数网格（如 0.1 ms 一格）
#   每个模块 = 一组，组内每档配置 = 一件物品 (cost, gain)
#   dp[c] = 该预算下的最大总收益；分组背包 O(组数 x 预算 x 组内档数)
#   **必须声明的三个前提**：收益可加、代价可加、预算可离散化
#   （现实里三个都只是近似：模块间有交互、延迟不线性叠加 —— 说出来是加分项）
'''
print(RECIPE)
for token in ["状态定义", "遍历顺序", "答案位置", "交换论证", "Soft-NMS", "WBF",
              "一对一标签分配", "分组背包", "p99"]:
    assert token in RECIPE, token
print("✅ 检查单覆盖：DP 五步 / 脱困路径 / 贪心的证明与反例 / NMS 补丁阶梯 / 预算分配的归约")

### 小结

- **DP 五步里最贵的是第 ① 步和第 ⑤ 步。**状态定义是一句中文，不是一个数组；
  答案位置写错会给出「74% 的用例都对」的程序——本 notebook 用打家劫舍的两种状态定义把它复现了。
  拿到错误答案时的诊断顺序是：**念状态定义 → 手算 dp[1]/dp[2] → 查答案位置**。
- **不可达状态必须是 ±inf。**填 0 等于宣称「存在一个代价为 0 的方案」，
  于是「恰好装满 / 最少硬币」类题目会静默地给出错误答案（`coins=[2], amount=3` 返回 0）。
- **一维压缩后遍历方向决定题目本身**：容量倒序是 01 背包，正序是完全背包，
  同一份代码只差一个 `range` 的方向。压缩的代价是**失去方案回溯**。
- **贪心只有两种合法答案：给出交换论证，或给出反例。**
  区间调度按「最早结束」可证明最优（400 组随机用例 0 次失手），
  按「最早开始」失手 57 次、按「最短区间」失手 11 次；
  找零在 `[1,3,4]` 上从 amount=6 起就错，而 amount=5 恰好对——**反例往往不在最小用例里**。
- **NMS 就是最大权独立集的贪心近似。**一维退化情形下它等价于加权区间调度，
  可以用 DP 求出精确最优：教科书反例上贪心/最优 = 10/12 = 0.833，
  最坏情况族的比值趋于 0（**近似比无常数下界**），随机场景平均 0.98 但约 20% 的场景次优。
  真实失效模式是「合并框吃掉两块相邻牌子」——本 notebook 里贪心召回 0%、DP 召回 100%。
- **但不要说「应该换成 DP」。**二维 bbox 的冲突图是任意图，最大权独立集 NP-hard（实测 2^n 增长），
  而且「总分最大」只是代理目标。工程解法是 **Soft-NMS / WBF / 一对一标签分配**——
  把去重从推理期的组合优化挪到训练期的约束。

下一站：**模块 05 · 模拟面试与压力下的编码** —— 45 分钟怎么分、卡住了怎么说、
以及蓄水池采样这类 ML 岗真正高频的原题。